In [ ]:
!pip -q install arch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model

# --------- Parameters ---------
ROLLING_WINDOW = 21  # trading days ~ X month
GLD_PATH = 'GLD ETF Stock Price History.csv'
GVZ_PATH = 'CBOE Gold Volatitity Historical Data.csv'

# --------- Load GLD ----------
gld = pd.read_csv(GLD_PATH)
gld.columns = [c.strip() for c in gld.columns]
date_col, price_col = 'Date', 'Price'

gld[date_col] = pd.to_datetime(gld[date_col])
gld = gld.sort_values(date_col)
gld = gld[[date_col, price_col]].dropna()

# log returns (decimal)
gld['ret'] = np.log(gld[price_col]).diff()

# realized volatility (annualized, decimal)
gld['rv'] = gld['ret'].rolling(ROLLING_WINDOW).std() * np.sqrt(252)

# --------- GARCH(1,1) on daily returns ----------
# Feed percent returns to arch for numerical stability
ret_pct = (gld['ret'].dropna() * 100)
am = arch_model(ret_pct, mean='Constant', vol='GARCH', p=1, q=1, dist='t')
res = am.fit(disp='off')

# res.conditional_volatility is DAILY σ in PERCENT, indexed to ret_pct dates
garch_daily_pct = res.conditional_volatility
# Convert to ANNUALIZED DECIMAL
garch_annual_dec = (garch_daily_pct / 100.0) * np.sqrt(252)

# Align back to GLD dates
gld['garch'] = np.nan
gld.loc[garch_annual_dec.index, 'garch'] = garch_annual_dec.values

# --------- Load GVZ (implied vol proxy) ----------
gvz = pd.read_csv(GVZ_PATH)
gvz.columns = [c.strip().lower() for c in gvz.columns]
date_col_gvz = [c for c in gvz.columns if 'date' in c][0]
iv_col_gvz = [c for c in gvz.columns if ('close' in c) or ('price' in c)][0]

gvz[date_col_gvz] = pd.to_datetime(gvz[date_col_gvz])
gvz = gvz.sort_values(date_col_gvz)
# GVZ is a percent; convert to decimal
gvz['iv'] = gvz[iv_col_gvz] / 100.0

# --------- Merge on date ----------
merged = pd.merge(
    gld[[date_col, 'rv', 'garch']],
    gvz[[date_col_gvz, 'iv']],
    left_on=date_col,
    right_on=date_col_gvz,
    how='inner'
).drop(columns=[date_col_gvz])

# Optional: drop initial NaNs from rolling window/GARCH warmup
merged = merged.dropna(subset=['rv', 'garch', 'iv'])

# --------- Plots ----------
plt.figure(figsize=(12,6))
plt.plot(merged[date_col], merged['rv'], label='Realized Vol (21d rolling)')
plt.plot(merged[date_col], merged['garch'], label='GARCH(1,1) Vol')
plt.plot(merged[date_col], merged['iv'], label='Implied Vol (GVZ)')
plt.title('Gold Volatility: Realized vs GARCH vs Implied (GVZ)')
plt.xlabel('Date'); plt.ylabel('Annualized Volatility (decimal)')
plt.grid(True); plt.legend(); plt.show()

# --------- Diagnostics ----------
corr = merged[['rv', 'garch', 'iv']].corr()
print("Correlation matrix:\n", corr, "\n")

mean_spread = (merged['iv'] - merged['rv']).mean()
print(f"Mean (GVZ - Realized) spread: {mean_spread:.4f}")
print("IV premium" if mean_spread > 0 else "IV discount / potential long-vol edge")
